# Atividade: Assistente de IA Generativa com Hugging Face e Gemini

**Disciplina:** Disruptive Architectures: IoT, IoB & Generative AI

**Grupo:**
- Enzo Monteiro Maciel - RM563734
- Matheus de Almeida Sousa - RM563557
- Paulo Estalise - RM563811
- Gabriel Bebé Silva - RM562012
- Emanuel Italo - RM561337

**Tema escolhido:** Tutor de Protocolos IoT

---

Neste notebook o grupo vai construir um assistente de IA em 4 etapas (e 1 bônus):

1. Assistente com personalidade (Hugging Face)
2. Comparação Hugging Face x Gemini
3. Chat com memória
4. Interface web com Gradio
5. (Bônus) API com FastAPI

Os trechos marcados com **`# >>> PERSONALIZE`** devem ser alterados pelo grupo.
Execute as células em ordem, de cima para baixo.

## 0. Configuração

In [1]:
# huggingface_hub -> cliente para chamar modelos remotamente (API do Hugging Face)
# google-genai    -> SDK oficial do Google Gemini
# gradio          -> cria interfaces web a partir de funções Python
!pip install huggingface_hub google-genai gradio -q

In [2]:
from huggingface_hub import InferenceClient
from google import genai
from google.genai import types
from google.colab import userdata

# Os tokens ficam nos Secrets do Colab (ícone de chave na barra lateral)
HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN ok" if HF_TOKEN else "ERRO: adicione HF_TOKEN nos Secrets do Colab")

HF_TOKEN ok


In [3]:
# Modelos usados na atividade (os mesmos da Aula 05)
MODELO_HF = "meta-llama/Llama-3.1-8B-Instruct"

cliente_hf = InferenceClient(model=MODELO_HF, token=HF_TOKEN, provider="auto")

In [4]:
# >>> PERSONALIZE: descreva o assistente do grupo de acordo com o tema escolhido.
# Este é o "system": a instrução que define o comportamento do assistente.
# Tema 3: Tutor de Protocolos IoT

SYSTEM_PROMPT = """Você é um tutor especializado em protocolos de comunicação IoT.
Explique MQTT, HTTP, CoAP e LoRa para iniciantes, de forma clara e com exemplos práticos.
Respostas em português, no máximo 5 frases.
Se a pergunta não for sobre protocolos IoT, diga educadamente que não pode ajudar."""

---
## Etapa 1: Assistente com personalidade (Hugging Face)

Cada mensagem enviada ao modelo tem uma etiqueta `role` que diz quem escreveu:

- `system`: a regra que o assistente deve seguir
- `user`: a pergunta do usuário

Nesta etapa, a **mesma pergunta** é enviada três vezes, e **só o `system` muda**:

| Chamada | system | user |
|---|---|---|
| 1 | Especialista no tema do grupo | mesma pergunta |
| 2 | Professor para crianças | mesma pergunta |
| 3 | Resposta em uma frase | mesma pergunta |

Se as respostas saírem diferentes, a diferença veio só do `system`. É assim que se criam assistentes diferentes em cima do mesmo modelo.

In [5]:
def perguntar_hf(pergunta, system, temperatura=0.7):
    """Envia uma pergunta ao modelo do Hugging Face e devolve o texto da resposta."""
    mensagens = [
        {"role": "system", "content": system},   # como o assistente deve se comportar
        {"role": "user",   "content": pergunta}, # o que o usuário perguntou
    ]
    resposta = cliente_hf.chat_completion(
        messages=mensagens,
        max_tokens=300,
        temperature=temperatura,
    )
    return resposta.choices[0].message.content

In [9]:
# >>> PERSONALIZE: crie 3 personalidades diferentes para o assistente do grupo.
personalidades = {
    "Especialista técnico": """Você é um engenheiro de IoT com anos de experiência em protocolos de comunicação.
    Explique MQTT, HTTP, CoAP e LoRa com detalhes técnicos, mencionando latência, throughput e casos de uso.
    Respostas em português, concisas.""",

    "Professor para iniciantes": """Você explica protocolos IoT como se estivesse ensinando para um aluno de engenharia no primeiro ano.
    Use analogias com coisas do dia a dia (como correio, telefone, etc) para tornar fácil de entender.
    Respostas em português, com exemplos simples.""",

    "Resumo em uma frase": """Você responde qualquer pergunta sobre protocolos IoT em uma única frase bem estruturada.
    Seja direto e conciso, em português.""",
}

# >>> PERSONALIZE: uma pergunta relacionada ao tema do grupo.
pergunta = "Qual a diferença entre MQTT e HTTP para comunicação IoT?"

for nome, system in personalidades.items():
    print(f"===== {nome} =====")
    print(perguntar_hf(pergunta, system))
    print()

===== Especialista técnico =====
MQTT e HTTP são protocolos de comunicação utilizados em IoT, mas com características diferentes:

**MQTT (Message Queuing Telemetry Transport)**

*   Protocolo leve e eficiente, projetado para comunicação de baixo consumo de banda.
*   Utiliza uma arquitetura de mensageiro (broker) para conectar dispositivos remotos.
*   Latência baixa (geralmente abaixo de 1s).
*   Throughput elevado (até 1000 mensagens por segundo).
*   Casos de uso: Sensores, atuadores, sistemas de monitoramento, automação industrial, veículos autônomos.

**HTTP (Hypertext Transfer Protocol)**

*   Protocolo de comunicação mais comum, projetado para web.
*   Utiliza uma conexão de síncrona e síncrona para troca de dados.
*   Latência maior (geralmente acima de 1s).
*   Throughput baixo (até 100 mensagens por segundo).
*   Casos de uso: Acesso web, aplicativos móveis, serviços de nuvem, etc.

Em resumo, MQTT é mais adequado para comunicação IoT, especialmente em casos de baixo consumo

**Observações do grupo (Etapa 1):**

- O que mudou nas respostas de cada personalidade?
- Qual `system` gerou a resposta mais útil para o tema? Por quê?

Não conseguimos gerar o token pelo Gemini!

---
## Etapa 2: Hugging Face x Gemini

Agora as mesmas perguntas vão para **dois modelos diferentes**.

No Gemini, o `system` não vai dentro da lista de mensagens. Ele é passado no parâmetro `system_instruction`.

In [10]:
# >>> PERSONALIZE: 3 perguntas sobre o tema do grupo.
perguntas = [
    "O que é MQTT e quando devo usar em projetos IoT?",
    "Qual a diferença entre CoAP e HTTP em termos de consumo de energia?",
    "Como o LoRa é diferente do Wi-Fi para comunicação de longo alcance?",
]

for p in perguntas:
    print("PERGUNTA:", p)
    print("\n--- Hugging Face (Llama) ---")
    print(perguntar_hf(p, SYSTEM_PROMPT))


PERGUNTA: O que é MQTT e quando devo usar em projetos IoT?

--- Hugging Face (Llama) ---
MQTT é um protocolo de comunicação de baixo nível, leve e eficiente para redes de sensores e dispositivos IoT. Ele permite a troca de pequenos pacotes de dados entre dispositivos e servidores em tempo real.

Use MQTT quando você precisar de uma comunicação escalável, segura e confiável entre dispositivos IoT, especialmente em aplicações que exigem baixo consumo de recursos e alta latência, como monitoramento de temperatura em um campo de sensores.

Exemplo: você pode usar MQTT para conectar sensores de temperatura em um campo de sensores para enviar dados de temperatura em tempo real a um servidor de IoT.

Exemplo prático: um sistema de monitoramento de temperatura em um campo de sensores utiliza MQTT para enviar dados de temperatura em tempo real a um servidor de IoT, que os analisa e envia alertas de temperatura elevada para os responsáveis.

O que você acha da MQTT?
PERGUNTA: Qual a diferença en

**Observações do grupo (Etapa 2):**

- Os dois modelos seguiram as regras do `SYSTEM_PROMPT` (idioma, tamanho, tema)?
- Qual respondeu melhor? Em qual pergunta a diferença foi maior?

Não conseguimos gerar o token pelo Gemini!

---
## Etapa 3: Chat com memória

O modelo **não guarda memória** entre uma chamada e outra.
Quem guarda a conversa é o nosso código, na lista `historico`, que é enviada inteira a cada mensagem.

Comandos do chat:
- `sair` encerra o chat
- `limpar` apaga o histórico (o assistente "esquece" a conversa)
- `historico` mostra quantas mensagens estão guardadas

**Teste sugerido:** diga seu nome, pergunte "qual é o meu nome?", digite `limpar` e pergunte de novo.

In [14]:
historico = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Chat iniciado! Comandos: sair | limpar | historico\n")

while True:
    entrada = input("Você: ")

    if entrada.lower() == "sair":
        print("Encerrando chat.")
        break

    if entrada.lower() == "limpar":
        historico = [{"role": "system", "content": SYSTEM_PROMPT}]  # mantém só o system
        print("\n(histórico apagado)\n")
        continue

    if entrada.lower() == "historico":
        print(f"\n(mensagens no histórico: {len(historico)})\n")
        continue

    historico.append({"role": "user", "content": entrada})

    resposta = cliente_hf.chat_completion(messages=historico, max_tokens=300)
    texto = resposta.choices[0].message.content

    historico.append({"role": "assistant", "content": texto})

    print(f"\nAssistente: {texto}\n")

Chat iniciado! Comandos: sair | limpar | historico

Você: qual é o meu nome

Assistente: Eu não posso fornecer informações sobre você. Posso ajudá-lo com protocolos de comunicação IoT?

Você: sair
Encerrando chat.


In [12]:
# Veja como ficou a lista enviada ao modelo
for msg in historico:
    print(f"[{msg['role']}] {msg['content'][:80].replace(chr(10), ' ')}")

[system] Você é um tutor especializado em protocolos de comunicação IoT. Explique MQTT, H
[user] Meu nome é Matheus
[assistant] Bem-vindo, Matheus! É um prazer ajudá-lo a entender os protocolos de comunicação


**Observações do grupo (Etapa 3):**

- O que aconteceu quando vocês perguntaram o nome antes e depois do `limpar`?
- Explique, com suas palavras, por que isso acontece.

Ele não informou meu nome, pois isso não é uma informação que foi ingerida na base de conhecimento. Dizer meu nome antes de eu informa-lo seria considerado alucinação.

---
## Etapa 4: Interface web com Gradio

O assistente ganha uma interface web, com a opção de escolher o modelo (Hugging Face ou Gemini).

O Gradio entrega o histórico da conversa já no formato de `role`/`content`.
A função `texto_da_mensagem` existe porque, dependendo da versão do Gradio, o conteúdo chega como texto simples ou como lista.

In [15]:
import gradio as gr

def texto_da_mensagem(conteudo):
    """Extrai o texto de uma mensagem do histórico do Gradio."""
    if isinstance(conteudo, str):
        return conteudo
    if isinstance(conteudo, list):
        return " ".join(item.get("text", "") for item in conteudo if isinstance(item, dict))
    return str(conteudo)


def responder(mensagem, historico_gradio, provedor):
    if provedor == "Hugging Face":
        # Formato HF: roles system, user e assistant
        mensagens = [{"role": "system", "content": SYSTEM_PROMPT}]
        for msg in historico_gradio:
            mensagens.append({"role": msg["role"], "content": texto_da_mensagem(msg["content"])})
        mensagens.append({"role": "user", "content": mensagem})

        resposta = cliente_hf.chat_completion(messages=mensagens, max_tokens=300)
        return resposta.choices[0].message.content

    else:
        # Formato Gemini: roles user e model; o system vai em system_instruction
        conteudos = []
        for msg in historico_gradio:
            papel = "model" if msg["role"] == "assistant" else "user"
            conteudos.append({"role": papel, "parts": [{"text": texto_da_mensagem(msg["content"])}]})
        conteudos.append({"role": "user", "parts": [{"text": mensagem}]})

        resposta = cliente_gemini.models.generate_content(
            model=MODELO_GEMINI,
            contents=conteudos,
            config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, max_output_tokens=300),
        )
        return resposta.text

In [16]:
# >>> PERSONALIZE: título, descrição e exemplos de acordo com o tema do grupo.
gr.ChatInterface(
    fn=responder,
    additional_inputs=[gr.Radio(["Hugging Face", "Gemini"], value="Hugging Face", label="Modelo")],
    title="🌐 Tutor de Protocolos IoT",
    description="Aprenda sobre MQTT, HTTP, CoAP e LoRa com um tutor de IA. Pergunte sobre protocolos de comunicação IoT e receba explicações claras para iniciantes.",
    examples=[
        ["O que é MQTT e quando devo usar?", "Hugging Face"],
        ["Qual a diferença entre CoAP e HTTP?", "Gemini"],
        ["Como funciona o LoRa?", "Hugging Face"],
    ],
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://118db8b8dfc0b9a2e8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Observações do grupo (Etapa 4):**

- Tire um print da interface funcionando e coloque no README do repositório do grupo.
- Troque de modelo no meio da conversa. O assistente continuou lembrando do que foi dito? Por quê?

Não conseguimos trocar de modelo pois não foi possivel gerar um token da API do Gemini.

> Para parar a interface, interrompa a célula (botão de parar do Colab).

---
## Etapa 5 (Bônus): API com FastAPI

Aqui o assistente vira uma **API REST**, como no final da Aula 05.
Qualquer frontend (site, app, dispositivo IoT) poderia chamar esse endpoint.

A célula abaixo cria o arquivo `app.py`.

In [18]:
%%writefile app.py
import os
from fastapi import FastAPI
from pydantic import BaseModel
from huggingface_hub import InferenceClient

# O token e o system prompt vêm de variáveis de ambiente (nunca escreva o token no código)
client = InferenceClient(
    model="meta-llama/Llama-3.1-8B-Instruct",
    token=os.environ["HF_TOKEN"],
    provider="auto",
)
SYSTEM_PROMPT = os.environ.get("SYSTEM_PROMPT", """Você é um tutor especializado em protocolos de comunicação IoT.
Explique MQTT, HTTP, CoAP e LoRa para iniciantes, de forma clara e com exemplos práticos.
Respostas em português, no máximo 5 frases.
Se a pergunta não for sobre protocolos IoT, diga educadamente que não pode ajudar.""")

app = FastAPI()

class Pergunta(BaseModel):
    mensagem: str

@app.post("/chat")
def chat(pergunta: Pergunta):
    resposta = client.chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": pergunta.mensagem},
        ],
        max_tokens=300,
    )
    return {"resposta": resposta.choices[0].message.content}

Overwriting app.py


In [19]:
# Inicia o servidor em segundo plano, dentro do próprio Colab
!pip install fastapi uvicorn -q

import os, subprocess, time
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["SYSTEM_PROMPT"] = SYSTEM_PROMPT

servidor = subprocess.Popen(["uvicorn", "app:app", "--port", "8000"])
time.sleep(5)  # espera o servidor subir
print("Servidor rodando em http://localhost:8000")

Servidor rodando em http://localhost:8000


In [20]:
# Testa o endpoint como um frontend faria (HTTP POST com JSON)
import requests

r = requests.post("http://localhost:8000/chat", json={"mensagem": "O que é MQTT?"})
print(r.status_code)
print(r.json()["resposta"])

200
MQTT (Message Queuing Telemetry Transport) é um protocolo de comunicação de baixo nível, leve e eficiente, projetado para enviar pequenos pacotes de dados entre dispositivos em redes de baixa largura de banda. É muito utilizado em IoT para conectar dispositivos que precisam enviar e receber dados em tempo real. Por exemplo, um sensor de temperatura pode enviar dados ao nuvem usando MQTT.


In [21]:
# Encerra o servidor
servidor.terminate()
print("Servidor encerrado.")

Servidor encerrado.
